# Objetivo
Analisar se há correlação entre os municípios com maior incidência de esquistossomose, em intervalos de cinco em cinco anos, e a presença de corpos d'água e variáveis climáticas.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("../../data/sinan_esqu_unsup.parquet")

if not data_path.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {data_path.resolve()}")

df = pd.read_parquet(data_path)

print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
df.head()

In [ ]:
import geobr
import pandas as pd

# baixar tabela de municípios
municipios = geobr.read_municipality()

municipios.head()

In [ ]:
# limpar código do município do geobr
municipios["code_muni"] = (
    municipios["code_muni"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str[:6]
)

# garantir mesmo tipo
df["ID_MUNICIP"] = df["ID_MUNICIP"].astype(str)
df["SG_UF"] = df["SG_UF"].astype(int)

# juntar informações
df = df.merge(
    municipios[
        ["code_muni", "name_muni", "code_state", "abbrev_state"]
    ],
    left_on="ID_MUNICIP",
    right_on="code_muni",
    how="left"
)

# criar colunas novas (sem apagar códigos)
df["municipio"] = df["name_muni"]
df["estado"] = df["abbrev_state"]

# remover apenas colunas temporárias
df = df.drop(
    columns=[
        "code_muni",
        "code_state",
        "name_muni",
        "abbrev_state"
    ]
)

df.head()

In [ ]:
import pandas as pd
import plotly.express as px
import geobr

# garantir formato de data
df["DT_NOTIFIC"] = pd.to_datetime(df["DT_NOTIFIC"])

# garantir tipo de AN_QUALI
df["AN_QUALI"] = pd.to_numeric(
    df["AN_QUALI"],
    errors="coerce"
)

# filtrar:
# AN_QUALI = 1
# período 2010-2014
casos = df[
    (df["AN_QUALI"] == 1) &
    (df["DT_NOTIFIC"].dt.year.between(2010, 2014))
].copy()

# manter formato consistente
casos["ID_MUNICIP"] = (
    casos["ID_MUNICIP"]
    .astype(str)
)

# carregar municípios
municipios = geobr.read_municipality()

# limpar código do geobr
municipios["code_muni"] = (
    municipios["code_muni"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str[:6]
)

# contar casos por município
casos_municipio = (
    casos.groupby("ID_MUNICIP")
    .size()
    .reset_index(name="qtd_casos")
)

# juntar geometrias
geo = municipios.merge(
    casos_municipio,
    left_on="code_muni",
    right_on="ID_MUNICIP",
    how="left"
)

geo["qtd_casos"] = geo["qtd_casos"].fillna(0)

# mapa
fig = px.choropleth(
    geo,
    geojson=geo.geometry,
    locations=geo.index,
    color="qtd_casos",
    hover_name="name_muni",
    hover_data={
        "abbrev_state": True,
        "qtd_casos": True
    },
    title="Casos positivos (AN_QUALI = 1) | 2010–2014"
)

fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    height=800
)

fig.show()

Queremos saber se os municipios com mais casos de esquistossomose no périodo de 5 anos, tem relação com a presença e distância de corpos d'água

Vamos usar os dados da plataforma Map Biomas: https://brasil.mapbiomas.org/estatisticas

In [ ]:
import pandas as pd

arquivo = r"C:\Users\anapd\Downloads\STATISTICS_MAPBIOMAS_AGUA_COL4_CITY-STATE-BIOME-SUB_BASINS.xlsx"

xls = pd.ExcelFile(arquivo)

print(xls.sheet_names)

In [ ]:
agua = pd.read_excel(
    arquivo,
    sheet_name="WATER_CITY_ANNUAL"
)

agua.head()

print(agua.columns.tolist())

In [ ]:
import pandas as pd

# carregar aba correta
agua = pd.read_excel(
    arquivo,
    sheet_name="WATER_CITY_ANNUAL"
)

# ----------------------------
# preparar água (2010–2014)
# ----------------------------

agua_5anos = (
    agua[
        agua["year"].between(2010, 2014)
    ]
    .groupby(
        ["code", "municipality", "state"]
    )
    .agg(
        area_agua_ha=("area_ha", "mean")
    )
    .reset_index()
)

# transformar código em texto
agua_5anos["code"] = (
    agua_5anos["code"]
    .astype(str)
)

# compatibilizar com SINAN (6 dígitos)
agua_5anos["code"] = (
    agua_5anos["code"]
    .str[:6]
)

# ----------------------------
# juntar
# ----------------------------

resultado = casos_municipio.merge(
    agua_5anos,
    left_on="ID_MUNICIP",
    right_on="code",
    how="left"
)
uf = {
    "Rondônia":"RO",
    "Acre":"AC",
    "Amazonas":"AM",
    "Roraima":"RR",
    "Pará":"PA",
    "Amapá":"AP",
    "Tocantins":"TO",
    "Maranhão":"MA",
    "Piauí":"PI",
    "Ceará":"CE",
    "Rio Grande do Norte":"RN",
    "Paraíba":"PB",
    "Pernambuco":"PE",
    "Alagoas":"AL",
    "Sergipe":"SE",
    "Bahia":"BA",
    "Minas Gerais":"MG",
    "Espírito Santo":"ES",
    "Rio de Janeiro":"RJ",
    "São Paulo":"SP",
    "Paraná":"PR",
    "Santa Catarina":"SC",
    "Rio Grande do Sul":"RS",
    "Mato Grosso do Sul":"MS",
    "Mato Grosso":"MT",
    "Goiás":"GO",
    "Distrito Federal":"DF"
}

resultado["UF"] = resultado["state"].map(uf)

# reorganizar colunas
resultado = resultado[
    [
        "ID_MUNICIP",
        "municipality",
        "UF",
        "qtd_casos",
        "area_agua_ha"
    ]
]

resultado.head()

In [ ]:
print(resultado["ID_MUNICIP"].duplicated().sum())

In [ ]:
print(resultado.isna().sum())


In [ ]:
print(
    resultado[
        ["qtd_casos","area_agua_ha"]
    ].corr()
)

In [ ]:
agua_tipo = pd.read_excel(
    arquivo,
    sheet_name="WATER_CITY_TYPE"
)

print(
    agua_tipo.columns
)

print(
    agua_tipo.iloc[:, :10].head()
)

In [ ]:
agua_tipo = pd.read_excel(
    arquivo,
    sheet_name="WATER_CITY_TYPE"
)

# média do período
agua_tipo_5anos = (
    agua_tipo[
        agua_tipo["year"].between(2010, 2014)
    ]
    .groupby(
        ["code","municipality","state"]
    )
    .agg(
        natural_ha=("natural_area_ha","mean"),
        anthropic_ha=("anthropic_area_ha","mean"),
        aquaculture_ha=("aquaculture_area_ha","mean")
    )
    .reset_index()
)

# compatibilizar código
agua_tipo_5anos["code"] = (
    agua_tipo_5anos["code"]
    .astype(str)
    .str[:6]
)

# juntar
resultado2 = casos_municipio.merge(
    agua_tipo_5anos,
    left_on="ID_MUNICIP",
    right_on="code",
    how="left"
)

resultado2.head()

In [ ]:
resultado2[
    [
        "qtd_casos",
        "natural_ha",
        "anthropic_ha",
        "aquaculture_ha"
    ]
].corr()

In [ ]:
fig = px.scatter(
    resultado2,
    x="anthropic_ha",
    y="qtd_casos",
    hover_name="municipality",
    log_x=True,
    log_y=True
)

fig.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# -- 1. Carregar populacao UFs -------------------------------------------------
pop_ufs_dir = Path("../../data/populacao/ufs")
pop_files = sorted(pop_ufs_dir.glob("pop_ufs_*.parquet"))
if not pop_files:
    raise FileNotFoundError(f"Nenhum parquet encontrado em: {pop_ufs_dir.resolve()}")

pop_ufs_list = []
for file_path in pop_files:
    year = int(file_path.stem.split("_")[-1])
    df_pop = pd.read_parquet(file_path)
    if "ano" not in df_pop.columns:
        df_pop["ano"] = year
    pop_ufs_list.append(df_pop)

pop_ufs = pd.concat(pop_ufs_list, ignore_index=True)

# -- 2. Normalizar tipos e chaves (SEM geobr) ---------------------------------
# Forçamos o 'code_state' e o 'ano' para números inteiros (Int64)
pop_ufs["ano"] = pd.to_numeric(pop_ufs["ano"], errors="coerce").astype("Int64")
pop_ufs["code_state"] = pd.to_numeric(pop_ufs["code_state"], errors="coerce").astype("Int64")

anos_populacao = sorted(pop_ufs["ano"].dropna().astype(int).unique().tolist())
if not anos_populacao:
    raise ValueError("Nenhum ano valido encontrado em pop_ufs.")

# Diagnostico rapido
print(f"Diagnostico pop_ufs: linhas={len(pop_ufs):,} | code_state nulos={pop_ufs['code_state'].isna().mean():.1%}")
if len(anos_populacao) > 5:
    print(f"Anos populacao: {anos_populacao[:5]} ... {anos_populacao[-5:]}")
else:
    print(f"Anos populacao: {anos_populacao}")

# -- 3. Heuristica: ano de populacao mais proximo do ano do caso --------------
def ano_mais_proximo(ano_caso: int, anos_disponiveis: list[int]) -> int:
    return min(anos_disponiveis, key=lambda a: abs(a - ano_caso))

# -- 4. Contar casos por ano e UF ---------------------------------------------
df_ano = df.copy()
df_ano["ano"] = df_ano["DT_NOTIFIC"].dt.year

# Forçamos o 'SG_UF' (que tem os códigos numéricos) e o 'ano' para Int64 também!
df_ano["SG_UF"] = pd.to_numeric(df_ano["SG_UF"], errors="coerce").astype("Int64")
df_ano["ano"] = df_ano["ano"].astype("Int64")

casos_uf = (
    df_ano.groupby(["ano", "SG_UF"])
    .size()
    .reset_index(name="casos")
)
print(f"Diagnostico casos_uf: linhas={len(casos_uf):,} | SG_UF nulos={casos_uf['SG_UF'].isna().mean():.1%}")

# -- 5. Mapear para o ano de populacao mais proximo ---------------------------
casos_uf["ano_pop"] = casos_uf["ano"].apply(
    lambda a: ano_mais_proximo(a, anos_populacao) if pd.notna(a) else pd.NA
)
casos_uf["ano_pop"] = casos_uf["ano_pop"].astype("Int64")

# -- 6. Merge com populacao usando ano_pop e o codigo da UF -------------------
# Adicionamos 'nome_uf' na seleção para trazer o nome completo do estado também
pop_ref = pop_ufs[["ano", "code_state", "populacao", "nome_uf"]].rename(
    columns={"ano": "ano_pop", "code_state": "SG_UF"}
)

inc_uf = casos_uf.merge(pop_ref, on=["ano_pop", "SG_UF"], how="left", indicator=True)

# Dicionário oficial do IBGE para gerar a 'uf_sigla' (2 letras)
mapa_siglas = {
    11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO',
    21: 'MA', 22: 'PI', 23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 28: 'SE', 29: 'BA',
    31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP',
    41: 'PR', 42: 'SC', 43: 'RS',
    50: 'MS', 51: 'MT', 52: 'GO', 53: 'DF'
}
# Criamos a nova coluna mapeando os códigos
inc_uf["uf_sigla"] = inc_uf["SG_UF"].map(mapa_siglas).astype("string")

missing_pop = inc_uf["populacao"].isna().mean()
print(f"Diagnostico merge: populacao nula={missing_pop:.1%}")

if missing_pop > 0:
    display(inc_uf[inc_uf["populacao"].isna()].head(10))

inc_uf = inc_uf.drop(columns=["_merge", "ano_pop"])
inc_uf["inc_100k"] = (inc_uf["casos"] / inc_uf["populacao"]) * 100_000

# Reorganizando as colunas para deixar o DataFrame elegante e pronto para o plot
ordem_colunas = ["ano", "SG_UF", "uf_sigla", "nome_uf", "casos", "populacao", "inc_100k"]
inc_uf = inc_uf[ordem_colunas].sort_values(["ano", "inc_100k"], ascending=[True, False]).reset_index(drop=True)

display(inc_uf.head(50))

In [ ]:
casos = df[
    (pd.to_numeric(df["AN_QUALI"], errors="coerce")==1)
    &
    (df["DT_NOTIFIC"].dt.year.between(2010,2014))
].copy()

casos["ano"] = (
    df["DT_NOTIFIC"].dt.year
)

casos["ID_MUNICIP"] = (
    casos["ID_MUNICIP"]
    .astype(str)
)

casos_municipio = (
    casos
    .groupby(
        ["ID_MUNICIP","ano"]
    )
    .size()
    .reset_index(name="casos")
)

In [ ]:
pop_municipios_path = Path("../../data/populacao/municipios/pop_municipios.parquet")

if not pop_municipios_path.exists():
    raise FileNotFoundError(
        f"Nenhum arquivo de população municipal encontrado em: {pop_municipios_path.resolve()}"
    )

pop_municipios = pd.read_parquet(pop_municipios_path)

pop_municipios["code_muni"] = (
    pop_municipios["code_muni"]
    .astype(str)
    .str[:6]
)
pop_municipios["ano"] = pd.to_numeric(
    pop_municipios["ano"],
    errors="coerce"
).astype("Int64")

inc_municipio = casos_municipio.merge(
    pop_municipios[
        [
            "code_muni",
            "ano",
            "populacao"
        ]
    ],
    left_on=[
        "ID_MUNICIP",
        "ano"
    ],
    right_on=[
        "code_muni",
        "ano"
    ],
    how="left"
)

inc_municipio["inc_100k"] = (
    inc_municipio["casos"]
    /
    inc_municipio["populacao"]
) * 100000

In [ ]:
resultado_final = inc_municipio.merge(
    resultado2[
        [
            "ID_MUNICIP",
            "municipality",
            "state",
            "natural_ha",
            "anthropic_ha",
            "aquaculture_ha"
        ]
    ],
    on="ID_MUNICIP",
    how="left"
)

resultado_final.head()